In [ ]:
!pip install pyreadstat


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 34.1 MB/s eta 0:00:00


In [12]:
import pandas as pd
import pyreadstat
import json
import xml.etree.ElementTree as ET
from sqlalchemy import create_engine
import pyarrow.feather as feather
import pyarrow.parquet as parquet
import pyarrow as pa
import os
class DataConverter:
    def __init__(self):
        self.input_formats = {
            'csv': pd.read_csv,
            'xls': pd.read_excel,
            'xlsx': pd.read_excel,
            'dta': pd.read_stata,
            'sav': lambda file_path: pyreadstat.read_sav(file_path)[0],
            'sas7bdat': lambda file_path: pyreadstat.read_sas7bdat(file_path)[0],
            'json': pd.read_json,
            'xml': self._read_xml,
            'feather': feather.read_feather,
            'parquet': lambda file_path: parquet.read_table(file_path).to_pandas()
        }

        self.output_formats = {
            'csv': lambda df, file_path: df.to_csv(file_path, index=False),
            'xlsx': lambda df, file_path: df.to_excel(file_path, index=False),
            'dta': lambda df, file_path: df.to_stata(file_path),
            'sav': lambda df, file_path: pyreadstat.write_sav(df, file_path),
            'json': lambda df, file_path: df.to_json(file_path, orient='records'),
            'xml': self._write_xml,
            'feather': lambda df, file_path: feather.write_feather(df, file_path),
            'parquet': lambda df, file_path: parquet.write_table(pa.Table.from_pandas(df), file_path),
            'pkl': lambda df, file_path: df.to_pickle(file_path),
            'sqlite': lambda df, file_path: df.to_sql('data', create_engine(f'sqlite:///{file_path}'), index=False, if_exists='replace')
        }

    def convert(self, input_file_path, output_file_type):
        ext_in = input_file_path.split('.')[-1].lower()

        if ext_in not in self.input_formats:
            raise ValueError(f"Unsupported input file format: {ext_in}")

        if output_file_type not in self.output_formats:
            raise ValueError(f"Unsupported output file format: {output_file_type}")

        # Generate output file path
        output_file_path = input_file_path.rsplit('.', 1)[0] + '.' + output_file_type
        # Read the data, note that this does not mean it will be clean or not throw and error
        df = self.input_formats[ext_in](input_file_path)
        self.output_formats[output_file_type](df, output_file_path)

        return output_file_path

    def _read_xml(self, file_path):
        tree = ET.parse(file_path)
        root = tree.getroot()
        data = []
        for child in root:
            record = {}
            for element in child:
                record[element.tag] = element.text
            data.append(record)
        return pd.DataFrame(data)

    def _write_xml(self, df, file_path):
        root = ET.Element('Data')
        for _, row in df.iterrows():
            record = ET.SubElement(root, 'Record')
            for col in df.columns:
                element = ET.SubElement(record, col)
                element.text = str(row[col])
        tree = ET.ElementTree(root)
        tree.write(file_path)
# Example usage
if __name__ == "__main__":
    converter = DataConverter()
    input_file = "/content/Y1.dta"
    output_file_type = "csv"
    output_file = converter.convert(input_file, output_file_type)
    print(f"Converted file saved as: {output_file}")


Converted file saved as: /content/Y1.csv
